In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
import collections
import types
from transformers.models.qwen3_5.modeling_qwen3_5 import Qwen3_5GatedDeltaNet
import copy

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-4B"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

In [ ]:
def get_post_hook(layer_name: str, cache_store):
    def post_hook(
        module, args, kwargs, output
    ):
        cache_params = copy.deepcopy(kwargs.get('cache_params', None)) if kwargs.get('cache_params', None) is not None else None
        cache_store.append(cache_params)
    return post_hook

In [ ]:
cache_store = []

for name, module in model.named_modules():
    if isinstance(module, Qwen3_5GatedDeltaNet) and "30" in name:
        module.register_forward_hook(get_post_hook(name, cache_store), with_kwargs=True)

In [ ]:
messages = [{"role": "user", "content": "Hi there! Who are you?"}]
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.inference_mode():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=20,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :]
response = tokenizer.decode(output_ids, skip_special_tokens=True)
print(response.strip())

In [ ]:
#so right now, cache_store is a list of 10 recurrent states.
#each recurrent state is a tensor of shape (1, 32, 128, 128)
# (1: batch size)
# (32: linear value heads)
# (128: per-head key / state dim)
# (128: per-head value dim)

In [ ]:
# Generate Deltas relative to the initial state:
def generate_delta(state, state_ref):
    return [state[i] - state_ref[i] for i in range(len(state)) if state[i] is not None]

#deltas shape: list of 9 lists, each cache_store[i] - cache_store[0].
#each inside list is a list of deltas, for each layer (24 total layers)
deltas = [generate_delta(cache_store[i].recurrent_states, cache_store[0].recurrent_states) for i in range(1, len(cache_store))]
torch_deltas = torch.stack([torch.stack(layer, dim=0) for layer in deltas], dim=0)
torch_deltas.shape

In [ ]:
S = torch.linalg.svdvals(torch_deltas)

In [ ]:
def get_s_energy(S, n = -1):
    # passed in batched S, shape (batch, head, dim). Want to do this over dim.
    S_energy = torch.sum(S ** 2, dim=-1, keepdim=True)
    S_nume = S ** 2
    if n == -1:
        return torch.sum(S_nume, dim=-1, keepdim=True) / S_energy
    else:
        return torch.sum(S_nume[..., :n], dim=-1, keepdim=True) / S_energy

In [ ]:
s_energy = get_s_energy(S, n=16)

In [ ]:
s_vals = torch.flatten(s_energy)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Convert to a flat CPU numpy array (works for torch tensors and numpy arrays)
if hasattr(s_vals, "detach"):
    data = s_vals.detach().float().cpu().numpy().ravel()
else:
    data = np.asarray([s_vals]).ravel()

data = data[np.isfinite(data)]

plt.figure(figsize=(7, 4))
plt.hist(data, bins=50, edgecolor="white", linewidth=0.6)
plt.title("Histogram of s_vals")
plt.xlabel("s_vals")
plt.ylabel("count")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()
